# Agentic Tool-Call Loop

Builds the core agent pattern: a loop where the LLM can call tools multiple times before producing a final answer. Then refactors into a reusable Agent class.

## Concepts covered
- **make_call**: dispatches LLM tool requests to real Python functions
- **Agentic (inner) loop**: keeps calling the LLM until it stops requesting tools
- **Q&A (outer) loop**: wraps the inner loop for multi-turn conversation with persistent message history
- **Multiple tools**: adding write access (`add_entry`) alongside read access (`search`)
- **Agent class**: encapsulates the loop pattern into a reusable object

In [1]:
from openai import OpenAI
openai_client = OpenAI()

In [2]:
import json
from gitsource import GithubRepositoryDataReader, chunk_documents
from minsearch import AppendableIndex

reader = GithubRepositoryDataReader(
    repo_owner="evidentlyai",
    repo_name="docs",
    allowed_extensions={"md", "mdx"},
)
files = reader.read()

parsed_docs = [doc.parse() for doc in files]
chunked_docs = chunk_documents(parsed_docs, size=3000, step=1500)

index = AppendableIndex(
    text_fields=["title", "description", "content"],
    keyword_fields=["filename"]
)
index.fit(chunked_docs)


In [3]:
def search(query):
    results = index.search(
        query=query,
        num_results=5
    )
    return results

search_tool = {
    "type": "function", 
    "name": "search",
    "description": "Search the documentation database for relevant results based on a query string.",
    "parameters": {
        "type": "object",
        "properties": {
            "query": {
                "type": "string",
                "description": "The search query to look up in the index"
            }
        },
        "required": [
            "query"
        ]
    }
}

In [ ]:
def make_call(tool_call):
    # Dispatch: parse the LLM's JSON arguments and route to the right Python function.
    # The LLM can't run code — it only outputs a name + arguments as JSON.
    # This function is the bridge between the LLM's request and actual execution.
    arguments = json.loads(tool_call.arguments)
    name = tool_call.name

    if name == 'search':
        result = search(**arguments)
    else:
        result = f'tool "{name}" not found'

    return {
        "type": "function_call_output",
        "call_id": tool_call.call_id,  # links this result back to the specific tool call
        "output": json.dumps(result),
    }

In [4]:
question = "How do I create a dahsbord in Evidently?"

In [37]:
instructions = """
You're a documentation assistant.

Answer the user question using the documentation knowledge base

IMPORTANT: When you explore the knowledge base, make at least 3 different 
searchers to make sure you explore the topic well.

Use only facts from the knowledge base when answering.
If you cannot find the answer, inform the user.

Our knowledge base is entirely about Evidently, 
so you don't need to include the word 'evidently'
in search results
"""

## Part 1: Single-Turn Agent (no loop yet)

The LLM is given tools and a question. We manually execute one round of tool calls
and feed the results back. This shows the mechanics before we wrap it in a loop.

In [38]:
message_history = [
    {"role": "system", "content": instructions},
    {"role": "user", "content": question}
]

In [20]:
response = openai_client.responses.create(
    model='gpt-4o-mini',
    input=message_history,
    tools=[search_tool]
)

In [21]:
message_history.extend(response.output)

In [22]:
message_history

[{'role': 'system',
  'content': "\nYou're a documentation assistant.\n\nAnswer the user question using the documentation knowledge base\n\nIMPORTANT: When you explore the knowledge base, make at least 3 different \nsearchers to make sure you explore the topic well.\n\nUse only facts from the knowledge base when answering.\nIMPORTANT: if you cannot find the answer, inform the user.\n"},
 {'role': 'user', 'content': 'How do I create a dahsbord in Evidently?'},
 ResponseFunctionToolCall(arguments='{"query":"create dashboard in Evidently"}', call_id='call_o6iZkLWLu6z8bZd5jMV8FNGa', name='search', type='function_call', id='fc_09030153ad66851b0069f6199b0d0c81958fc3b24114dd3ed4', namespace=None, status='completed'),
 ResponseFunctionToolCall(arguments='{"query":"Evidently dashboard features"}', call_id='call_V79iF5LfYRFkX28Be3ftcmRH', name='search', type='function_call', id='fc_09030153ad66851b0069f6199b0d1c8195a4bc0b7371a6fb24', namespace=None, status='completed'),
 ResponseFunctionToolCall

In [28]:
for message in response.output:
    if message.type == 'function_call':
        print(f'executing {message.name}({message.arguments})...')
        tool_call_output  = make_call(message)
        message_history.append(tool_call_output)

executing search({"query":"create dashboard in Evidently"})...
executing search({"query":"Evidently dashboard features"})...
executing search({"query":"Evidently dashboard example"})...


In [29]:
len(message_history)

8

In [31]:
response = openai_client.responses.create(
    model='gpt-4o-mini',
    input=message_history,
    tools=[search_tool]
)

response.usage.input_tokens

11471

In [36]:
print(response.output_text)

To create a dashboard in **Evidently**, follow the steps below:

### 1. **Set Up Your Project**
Before creating a dashboard, ensure that you've connected to **Evidently Cloud** and created a project that includes reports with evaluation results.

### 2. **Add a Dashboard**
You can create a dashboard in two ways: using the user interface or via the Python API.

#### **Using the User Interface:**
- **Enter Edit Mode**: Navigate to the dashboard and click on the "Edit" mode in the top right corner.
- **Add a Tab**:
  - Click the plus sign to add a new tab.
  - Optionally, choose "empty" to create a custom tab and provide a name.
- **Adding Panels**:
  - Click on "Add Panel."
  - Follow the prompts to choose your metrics and configure the panel.
  - Click "Save" and select the tab where you want to add the panel.

#### **Using Python API:**
If you prefer coding, you can add tabs and panels directly through the API:
- **Add a Tab**:
    ```python
    project.dashboard.add_tab("Your Tab Name

## Agentic tool-call loop

In [44]:
instructions = """
You're a documentation assistant.

Answer the user question using the documentation knowledge base

Make 3 iterations:
1) in the first iteration, perform one search
2) in the second iteration, analyze the results from the previous search
    and perform 2 more searches
3) syntesize the results into the output

IMPORTANT: At each step, give an explanation of why you want to perform
search for this particular search query. It should be 2-3 sentences explaining
the logic of your decision.


Use only facts from the knowledge base when answering.
If you cannot find the answer, inform the user.

Our knowledge base is entirely about Evidently, 
so you don't need to include the word 'evidently'
in search results
"""

question = "How do I create a dahsbord in Evidently?"

In [45]:
message_history = [
    {"role": "system", "content": instructions},
    {"role": "user", "content": question}
]

iteration_number = 1

while True:
    response = openai_client.responses.create(
        model='gpt-4o-mini',
        input=message_history,
        tools=[search_tool],
    )
    print(f'iteration number {iteration_number}...')

    # you can also add cost of iteration here
    
    
    message_history.extend(response.output)

    # we stop the loop if agent doesn't want to make a function call
    has_function_calls = False
    
    for message in response.output:
        if message.type == 'function_call':
            print(f'executing {message.name}({message.arguments})...')
            tool_call_output  = make_call(message)
            message_history.append(tool_call_output)
            has_function_calls = True
            
        if message.type == 'message':
            text = message.content[0].text
            print('ASSISTANT:', text)
    iteration_number += 1
    print()
    if not has_function_calls:
        break

    

iteration number 1...
ASSISTANT: To start, I'll search for information specifically on creating dashboards. This will provide foundational guidance on the dashboard creation process, including any necessary steps, tools, or features involved in the procedure. 

I'll proceed with the search now.
executing search({"query":"create a dashboard"})...

iteration number 2...
ASSISTANT: The initial search results provide a comprehensive overview of how to create a dashboard and add panels in the platform. They detail the steps for adding Tabs, defining Panels, and accessing various visualizations such as charts and counters.

To develop a more in-depth understanding, I will perform two additional searches: 
1. To explore specific visualizations and their configurations.
2. To find any prerequisites or settings for creating dashboards.

Let's proceed with these searches.
executing search({"query":"dashboard visualizations configurations"})...
executing search({"query":"dashboard prerequisites s

## ADDING A TOOL

In [55]:
def add_entry(filename, title, description, content):
    entry = {'start': 0,
              'content': content,
              'title': title,
              'description': description,
              'filename': filename
             }

    index.append(entry)
    return "OK"

add_entry_tool ={
   "type": "function",
   "name": "add_entry",
   "description": "Add a new entry with metadata and content to an index.",
   "parameters": {
      "type": "object",
      "properties": {
         "filename": {
            "type": "string",
            "description": "The name of the file associated with the entry"
         },
         "title": {
            "type": "string",
            "description": "The title of the entry"
         },
         "description": {
            "type": "string",
            "description": "A short description of the entry"
         },
         "content": {
            "type": "string",
            "description": "The main content of the entry"
         }
      },
      "required": [
         "filename",
         "title",
         "description",
         "content"
      ]
   }
}


In [56]:
def make_call(tool_call):
    arguments = json.loads(tool_call.arguments)
    name = tool_call.name

    if name == 'search':
        result = search(**arguments)
    elif name == 'add_entry':
        result = add_entry(**arguments) # iterating if there are available tools
    else:
        result = 'not found tool "{name}"'
    # return call output
    return {
        "type": "function_call_output",
        "call_id": tool_call.call_id,
        "output": json.dumps(result),
    }

In [57]:
message_history = [
    {"role": "system", "content": instructions},
    {"role": "user", "content": question}
]

iteration_number = 1

while True:
    response = openai_client.responses.create(
        model='gpt-4o-mini',
        input=message_history,
        tools=[search_tool, add_entry_tool],
    )
    print(f'iteration number {iteration_number}...')

    # you can also add cost of iteration here
    
    
    message_history.extend(response.output)

    # we stop the loop if agent doesn't want to make a function call
    has_function_calls = False
    
    for message in response.output:
        if message.type == 'function_call':
            print(f'executing {message.name}({message.arguments})...')
            tool_call_output  = make_call(message)
            message_history.append(tool_call_output)
            has_function_calls = True
            
        if message.type == 'message':
            text = message.content[0].text
            print('ASSISTANT:', text)
    iteration_number += 1
    print()
    if not has_function_calls:
        break

    

iteration number 1...
executing search({"query":"create dashboard"})...

iteration number 2...
ASSISTANT: To create a dashboard, you can start by adding panels to visualize evaluation results. Dashboards consist of several panels that can display various metrics over time, and it's important that you have already added reports with evaluation results to your project. Here’s a general approach to creating a dashboard:

1. **Add Tabs**: Enter edit mode and click the plus sign to add new tabs for organizing your panels. You can start with empty tabs or use pre-built templates.
2. **Add Panels**: Use the "Add Panel" button to include different types of visual representations (like text, counters, pie charts, line plots, etc.). Follow the prompts to configure each panel with the metrics relevant to your reports.

Now, let's delve deeper into the details of how to create a dashboard by conducting additional searches.
executing search({"query":"adding tabs to dashboard"})...
executing search(

In [59]:
message_history.append(
    {"role": "user", "content": "add this content to our database"}
)

In [ ]:
# We reuse the same loop here — the agent doesn't know or care that the user
# just asked it to write to the database. It just sees a new user message,
# runs the tool-call loop, and happens to choose add_entry this time.
# This is why the loop is a general pattern: it works for any combination of tools.

while True:
    response = openai_client.responses.create(
        model='gpt-4o-mini',
        input=message_history,
        tools=[search_tool, add_entry_tool],
    )
    print(f'iteration number {iteration_number}...')
    message_history.extend(response.output)

    has_function_calls = False
    
    for message in response.output:
        if message.type == 'function_call':
            print(f'executing {message.name}({message.arguments})...')
            tool_call_output = make_call(message)
            message_history.append(tool_call_output)
            has_function_calls = True
            
        if message.type == 'message':
            text = message.content[0].text
            print('ASSISTANT:', text)

    iteration_number += 1
    print()
    if not has_function_calls:
        break

In [61]:
index.docs[-1]

{'start': 0,
 'content': 'To create a dashboard in your project, you\'ll need to follow these steps:\n\n1. **Enable Edit Mode**: Begin by entering \'Edit\' mode on the dashboard. You can find this option in the top right corner of the dashboard interface.\n\n2. **Add Tabs**:\n   - Click the plus sign to add a new tab. You can either create an empty tab or select from pre-built templates to help organize your panels. Pre-built tabs rely on existing metrics, so ensure that relevant data is already included in your project.\n   - To delete a tab, re-enter \'Edit\' mode, click on the edit tabs icon next to the tab names, and select which tab to remove.\n\n3. **Create Panels**:\n   - Use the "Add Panel" button to include various types of visualizations (like text, counters, pie charts, line plots, etc.). \n   - You\'ll follow a prompt to configure each panel by selecting the metrics you want to display. Ensure that these metrics match those logged in your reports for accurate data visualiza

## Part 4: Q&A Loop

Wraps the agentic loop in an outer conversation loop so the user can ask multiple questions.
Message history is preserved across turns so the LLM remembers the conversation.

In [ ]:
tools = [search_tool, add_entry_tool]

message_history = [
    {"role": "system", "content": instructions}
]

# Outer loop: keeps the conversation going until the user types 'stop'.
# Each user turn runs the full inner agentic loop before asking for the next question.
while True:
    user_prompt = input('You: ')
    if user_prompt.lower().strip() == 'stop':
        break
    message_history.append({"role": "user", "content": user_prompt})

    # Inner loop: keeps calling the LLM until it stops requesting tools.
    while True:
        response = openai_client.responses.create(
            model='gpt-4o-mini',
            input=message_history,
            tools=tools,
        )
        message_history.extend(response.output)
        has_function_calls = False

        for message in response.output:
            if message.type == 'function_call':
                print(f'executing {message.name}({message.arguments})...')
                tool_call_output = make_call(message)
                message_history.append(tool_call_output)
                has_function_calls = True
            if message.type == 'message':
                print('ASSISTANT:', message.content[0].text)

        if not has_function_calls:
            break

## Part 5: Agent Class Refactor

Encapsulates the loop pattern into a reusable class.
- `loop(user_prompt, message_history)` — runs one turn (inner loop), returns updated history
- `qna()` — runs the full conversation (outer loop), calls `loop()` each turn

In [ ]:
import json

class Agent:

    def __init__(self, llm_client, model, instructions, tools):
        self.llm_client = llm_client
        self.model = model
        self.instructions = instructions
        self.tools = tools

    def make_call(self, tool_call):
        arguments = json.loads(tool_call.arguments)
        name = tool_call.name
        if name == 'search':
            result = search(**arguments)
        elif name == 'add_entry':
            result = add_entry(**arguments)
        else:
            result = f'tool "{name}" not found'
        return {
            "type": "function_call_output",
            "call_id": tool_call.call_id,
            "output": json.dumps(result),
        }

    def loop(self, user_prompt, message_history=None):
        # Start fresh history if none provided (single-turn usage).
        # If history is passed in, conversation context is preserved (multi-turn usage).
        if not message_history:
            message_history = [{"role": "system", "content": self.instructions}]

        message_history.append({"role": "user", "content": user_prompt})

        while True:
            response = self.llm_client.responses.create(
                model=self.model,
                input=message_history,
                tools=self.tools,
            )
            message_history.extend(response.output)
            has_function_calls = False

            for message in response.output:
                if message.type == 'function_call':
                    print(f'executing {message.name}({message.arguments})...')
                    tool_call_output = self.make_call(message)
                    message_history.append(tool_call_output)
                    has_function_calls = True
                if message.type == 'message':
                    print('ASSISTANT:', message.content[0].text)

            if not has_function_calls:
                break

        return message_history

    def qna(self):
        # Persist history across turns so the LLM remembers earlier questions.
        message_history = [{"role": "system", "content": self.instructions}]

        while True:
            user_prompt = input('You: ')
            if user_prompt.lower().strip() == 'stop':
                break
            message_history = self.loop(user_prompt, message_history)